# Descubrimiento rápido del arreglo de loops (WSI a ciegas, versión rápida), Colab L4

El #13 encontró solo un arreglo de loops en el Qwen3.5-0.8B (vía MoI empató con p50c), pero tardó **9.1 h**. WSI en sí tardó ~30 min; el resto fue calificar 60 configuraciones. Esta versión quita las tres causas:

1. **Corte de loops arreglado.** vLLM 0.34 cambió la API y el corte se apagaba solo; cada loop corría hasta 16k tokens.
2. **vLLM se carga una vez.** Los pesos editados se escriben en memoria entre configuraciones (con checksum exacto contra el checkpoint horneado).
3. **12 configuraciones en vez de 60** (1–2 capas × 16/todas las neuronas × α 4/8/12); las 3 mejores a la ronda 2.

WSI sigue decidiendo solo qué capas, qué neuronas y qué dirección. El examen (200 GSM8K de test a 16k) usa checkpoints horneados en vLLM nuevo, igual que el #13, con control de dirección al azar y MoI.

**Modelo:** `MODEL` abajo (por defecto **Qwen3.5-4B**, bf16). Si casi no hace loops en GSM8K, se detiene después del diagnóstico y lo dice.

**Uso:** `L4` → `Run all` → acepta Drive. ~2.5–4 h. **Si se desconecta, vuelve a darle Run all:** cada configuración terminada queda en `MyDrive/wsi_runs/discover_fast_<modelo>` y sigue desde ahí. El resultado queda en `MyDrive/wsi_runs/FINALES/discover_fast_<modelo>.json` y el runtime se apaga solo si ese archivo existe.

In [ ]:
MODEL = "Qwen/Qwen3.5-4B"          # o "Qwen/Qwen3.5-0.8B" para comparar con el #13
TAG = MODEL.split("/")[-1].lower().replace(".", "")
RUN_NAME = f"discover_fast_{TAG}"
REPO = "/content/wsi"
BASE = "/content/base"
W = "/content/work"
S = f"{REPO}/scripts/discover_fast.py"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm
!pip install -q flash-linear-attention || echo "fla opcional"
import vllm, transformers; print("vllm", vllm.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, glob, shutil, json, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
DRIVE = f"/content/drive/MyDrive/wsi_runs/{RUN_NAME}"
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
os.makedirs(DRIVE, exist_ok=True); os.makedirs(FINALES, exist_ok=True); os.makedirs(W, exist_ok=True)
for f in glob.glob(f"{DRIVE}/*"):           # lo que ya se terminó en una sesión anterior
    if not os.path.exists(f"{W}/{os.path.basename(f)}"):
        shutil.copy2(f, W)
os.environ["WSI_SYNC_DIR"] = DRIVE          # el script copia aquí cada configuración que termina
print("ya hecho:", sorted(os.listdir(DRIVE)) or "nada")

In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git pull -q
assert os.path.exists(S), f"{S} no está en GitHub todavía: falta hacer push"
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit", COMMIT)

def final(extra):
    # resultado a Drive FINALES: primero a un tmp, luego os.replace
    import datetime
    res = json.load(open(f"{W}/results_discover_fast.json")) if os.path.exists(f"{W}/results_discover_fast.json") else {}
    if os.path.exists(f"{W}/diagnosis_blind.json"):
        d = json.load(open(f"{W}/diagnosis_blind.json")); d.pop("neuron_rank", None)
        res.setdefault("diagnosis", d)
    if os.path.exists(f"{W}/timing_fast.json"):
        res.setdefault("timing_s", json.load(open(f"{W}/timing_fast.json")))
    res["_final"] = {"complete": True, "model": MODEL, "commit": COMMIT, "notebook": "discover_fast_colab.ipynb",
                     "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"), **extra}
    dst = f"{FINALES}/{RUN_NAME}.json"
    with open(dst + ".tmp", "w") as f:
        json.dump(res, f, indent=1)
    os.replace(dst + ".tmp", dst)
    print("guardado en", dst)

In [ ]:
# 1. problemas (GSM8K train para descubrir, test solo para el examen) y respuestas del modelo base
if not os.path.exists(f"{W}/config.json"):
    !python {S} prepare --work {W} --base-dir {BASE} --model-repo {MODEL}
!python -u {S} elicit --work {W}

In [ ]:
# 2. WSI: rankea TODAS las capas por separación loop/normal (d' en datos apartados) y ordena las neuronas.
# Si el modelo casi no hace loops, se detiene aquí (eso también es un resultado).
if not os.path.exists(f"{W}/diagnosis_blind.json"):
    r = subprocess.run([sys.executable, "-u", S, "diagnose", "--work", W], capture_output=True, text=True)
    print(r.stdout[-3000:]); print(r.stderr[-3000:])
    LOOPS_OK = r.returncode == 0
    if not LOOPS_OK:
        final({"status": "stopped_after_diagnose", "reason": (r.stderr.strip().splitlines() or ["?"])[-1]})
else:
    LOOPS_OK = True
    print("diagnóstico ya hecho")

In [ ]:
# 3. búsqueda: 12 configuraciones en 32 problemas, las 3 mejores en 64 más; se congela el método antes del examen
if LOOPS_OK:
    !python -u {S} search --work {W}
    !python {S} freeze --work {W} --git-commit {COMMIT}
    print(open(f"{W}/frozen.json").read())

In [ ]:
# 4. examen: 200 GSM8K test a 16k; base, arreglo siempre prendido, control al azar, y los dos vía MoI
if LOOPS_OK:
    !python -u {S} test --work {W} --ck-root /content
    !python {S} report --work {W}
    final({"status": "done"})

In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
import time
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(90)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")